In [70]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from sklearn.linear_model import LinearRegression
import yfinance as yf
import pandas_datareader.data as web



In [ ]:

msft_prices_manual = {
    "2024-12-01": 421.57,
    "2025-01-01": 425.53,
    "2025-02-01": 411.60,
    "2025-03-01": 398.82,
    "2025-04-01": 374.65,
    "2025-05-01": 431.11,
    "2025-06-01": 457.14,
    "2025-07-01": 496.47,
    "2025-08-01": 535.00,
    "2025-09-01": 500.47,
    "2025-10-01": 514.80,
    "2025-11-01": 519.81
}

msft_new = pd.DataFrame.from_dict(msft_prices_manual, orient='index', columns=['AdjClose'])
msft_new.index = pd.to_datetime(msft_new.index)
msft_new = msft_new.sort_index()

msft_new['MSFT'] = msft_new['AdjClose'].pct_change()

msft_new = msft_new[['MSFT']]
# data cut off is 2024 of new_data csv, so have to extend microsoft data for this year 

In [68]:
new_data=pd.read_csv('daat.csv') #2.2 million line df I use for project work, 
# has every US equity monthly price since 2000
new_data.drop(columns='PERMNO',inplace=True)
new_data.rename(columns={'date':'Date','TICKER':'Ticker'},inplace=True)
new_monthly_data=new_data.drop(columns='PRC') # daat.csv is basically every single US ticker and monthly prices since 2000, its 2.2 million lines so I am not going to submit it
new_monthly_data=new_monthly_data.pivot_table(index='Date', columns='Ticker', values='RET', aggfunc='first')   
msft_full = pd.concat([new_monthly_data, msft_new])

In [69]:
msft_full['MSFT']

2000-01-31             -0.161670
2000-02-29             -0.086845
2000-03-31              0.188811
2000-04-28             -0.343529
2000-05-31             -0.103047
                         ...    
2025-07-01 00:00:00     0.086035
2025-08-01 00:00:00     0.077608
2025-09-01 00:00:00    -0.064542
2025-10-01 00:00:00     0.028633
2025-11-01 00:00:00     0.009732
Name: MSFT, Length: 312, dtype: object

In [64]:
ff3 = gff.famaFrench3Factor(frequency='m')
ff3.rename(columns={"date_ff_factors": "Date"}, inplace=True)
ff3.set_index("Date", inplace=True)
ff3.index = ff3.index.to_period("M").to_timestamp("D")

In [ ]:
start = ff3.index.min()
end   = ff3.index.max()

vti = yf.download(
    "VTI",
    start=start,
    end=end,
    interval="1mo",
    auto_adjust=True,
    progress=False
)

vti["VTI_ret"] = vti["Adj Close"].pct_change()
vti.index = vti.index.to_period("M").to_timestamp("D")
vti_m = vti[["VTI_ret"]].dropna()
ff3_vti = ff3.join(vti_m, how="inner")
ff3_vti["Mkt-RF"] = ff3_vti["VTI_ret"] - ff3_vti["RF"] / 100.0


In [ ]:
msft = msft_full['MSFT'].dropna()
msft.index = pd.to_datetime(msft.index)
msft.index = msft.index.to_period("M").to_timestamp("D")
df = pd.concat([msft, ff3_vti[["Mkt-RF", "RF"]]], axis=1).dropna()
df['MSFT'] = pd.to_numeric(df['MSFT'], errors='coerce')
df['MSFT_excess'] = df['MSFT'] - df['RF']
betas = pd.DataFrame(columns=['Beta'])

for date in dates_2025:
    start = date - relativedelta(years=5)
    window = df.loc[start:date]
    X = window[['Mkt-RF']]
    y = window['MSFT_excess']

    model = LinearRegression().fit(X, y)

    betas.loc[date] = model.coef_[0]

In [76]:
betas

,Beta
2025-01-01,0.839380
2025-02-01,0.851504
2025-03-01,0.863775
2025-04-01,0.960465
2025-05-01,0.965581
2025-06-01,0.968638
2025-07-01,0.962028
2025-08-01,0.975717
2025-09-01,0.933900
2025-10-01,0.927206
